In [1]:
import sys, os
sys.path.insert(0, '../../utils')


In [2]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from connectome_types import CONNECTOME_SYN_TABLE_PATH
from utils import load_neurons_table, load_synapses_position_transformed
from neuron_custom_features import calc_spines_features
from activity_utils import require_activity_h5
from plot_utils import ex_color
from ensembles import (require_ensemble_results, build_shared_input_strength_series,
                       member_cell_type_order, member_cell_type_stats,
                       recorded_coverage_table, recorded_ex_root_ids)


In [3]:
# ── how this figure is drawn ─────────────────────────────────────────────────
# The panel drawing lives here rather than in utils/, so the notebook that produces
# figure S15 is readable end to end. utils/ensembles.py holds the analysis: detection,
# the matched controls and the bootstraps, all run once by scripts/ensemble_run.py.

import matplotlib
from matplotlib.colors import to_rgb


CONTROL_COLOR  = "#999999"       # histogram fill (rendered with alpha=0.7)


                                 # 999999 @ alpha 0.7 on white so it matches the hist
SPINY_COLOR    = '#7C3AED'       # synapse onto a spine


SHAFT_COLOR    = '#059669'       # synapse onto shaft/soma


_NULL_COLOR = '#999999'


# Cell types drawn in the spiny colour by `plot_member_cell_type_panel`. The IT-like
# types the ensemble detector actually returns in bulk; every other E type gets the
# aspiny colour, which is what figure 6's supplement contrasts.
SPINY_CELL_TYPES = ('23P', '4P', '5P-IT')


# (column key, header, format) of `recorded_coverage_table` as `plot_cell_type_table`
# draws it: the two counts and what fraction of the type the subset caught.
# Headers wrap where a single line would run into the neighbouring column.
COVERAGE_TABLE_COLUMNS = (('n_reference', 'Micro\ncolumn', '{:,.0f}'),
                          ('n_subset',    'Recorded',     '{:,.0f}'),
                          ('coverage',    'Recorded\n(%)', '{:.0%}'))


def darken(color, factor=0.65):
    """A darker shade of `color` — for lines that must read against their own fill."""
    r, g, b = to_rgb(color)
    return (r * factor, g * factor, b * factor)


def plot_null_hist_panel(ax, null_vals, obs_val, global_val, ctrl_color, obs_color,
                         obs_label, global_label, ctrl_label='Control',
                         bins=20, xlabel='', ylabel='Count', star=None,
                         legend_loc='upper left', legend_bbox=(0, 1.05),
                         headroom=1.5, marker_pad=1.03, ctrl_mean_color=None,
                         ctrl_mean_fmt='.3g', trim_yticks=True):
    """Null-distribution histogram + observed / control-mean vertical markers.

    Shared by the metric-B (connection probability) and metric-A (% synapses on
    spines) panels, which differ only in data, bins and x label.

    `ax.axvline` spans the full axes height, so with a `best`-placed legend the
    marker lines and the tallest bars run straight through the legend text. Here
    the markers are drawn with `vlines` capped just above the tallest bar
    (`marker_pad`) and the y limit is opened to `headroom` × that height, so the
    legend sits in a band that no artist reaches into.

    `star` (e.g. from `stats_corr.p_to_stars`) is drawn between the ensemble
    and control-mean vlines, including when it is 'ns'.

    The control-mean dashed line sits on top of the control bars, so drawing it
    in `ctrl_color` makes it near-invisible however opaque it is — same hue, and
    the bars behind it are only alpha-lightened. It is drawn in a darkened shade
    (`ctrl_mean_color`, default `darken(ctrl_color)`) so it reads as the same
    grey but stands off its own histogram.

    `trim_yticks` drops the y ticks that fall inside the headroom band. The band
    is deliberate — it is what keeps the legend off the bars — but no bar ever
    reaches it, so ticks up there label empty space. The limit is unchanged.

    Returns the histogram counts.
    """
    ctrl_mean = np.mean(null_vals)
    ctrl_leg_label = f'{ctrl_label} (mean = {ctrl_mean:{ctrl_mean_fmt}})'
    counts, _, _ = ax.hist(null_vals, bins=bins, color=ctrl_color, alpha=0.7,
                           label=ctrl_leg_label)
    y_top = counts.max() * marker_pad
    ax.vlines(obs_val, 0, y_top, color=obs_color, lw=2, label=obs_label)
    ax.vlines(global_val, 0, y_top, color='k', lw=1.5, ls='--', label=global_label)
    ax.vlines(ctrl_mean, 0, y_top, lw=2, ls='--',
              color=darken(ctrl_color) if ctrl_mean_color is None else ctrl_mean_color)
    ax.set_ylim(0, counts.max() * headroom)
    if trim_yticks:
        # set_yticks can rescale the view, so restore the limit afterwards —
        # the point is to lose the ticks, not the headroom they sat in.
        _ylim = ax.get_ylim()
        ax.set_yticks([t for t in ax.get_yticks() if 0 <= t <= counts.max()])
        ax.set_ylim(_ylim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    # Ensembles leads the legend. Draw order would put Control first — the
    # histogram has to be drawn before the vlines that cap to its height — but
    # ensembles is the result the panel is about, so the entries are reordered
    # rather than the drawing, which would change the z-order too.
    _h, _l = ax.get_legend_handles_labels()
    _by_label = dict(zip(_l, _h))
    _order = [lb for lb in (obs_label, ctrl_leg_label, global_label)
              if lb in _by_label]
    ax.legend([_by_label[lb] for lb in _order], _order,
              frameon=False, loc=legend_loc, bbox_to_anchor=legend_bbox)
    if star:
        x_lo = min(obs_val, ctrl_mean)
        x_hi = max(obs_val, ctrl_mean)
        x_mid = (x_lo + x_hi) / 2
        y_star = y_top
        ax.annotate('', xy=(x_hi, y_star), xytext=(x_lo, y_star),
                    arrowprops=dict(arrowstyle='-', color='black', lw=0.75, alpha=0.8),
                    annotation_clip=False)
        ax.text(x_mid, y_star, star, ha='center', va='bottom',
                fontsize=matplotlib.rcParams.get('font.size', 12), color='black',
                clip_on=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    return counts


def plot_member_cell_type_panel(ax, member_cell_type_df, spiny_color, aspiny_color,
                                order=None, spiny_types=SPINY_CELL_TYPES,
                                mode='fraction', reference_cell_types=None,
                                ylabel=None, bar_width=0.7, label_rotation=45,
                                spiny_types_are_prefix=False, z_sig_level=1.96,
                                reference_color=None, reference_width=0.9,
                                reference_label='Recorded population',
                                filled_label=None, filled_color='#999999',
                                drop_types=(), legend=False):
    """Cell-type composition of ensemble members on one axes — figure version.

    Differs from `plot_member_cell_types` (the exploratory grid in results_analysis)
    on the two things a paper panel needs: **distinct neurons**, not member slots — a
    neuron detected in two ensembles is one neuron here — and no title/legend, so the
    caption carries the labelling. Bars and their x tick labels share a colour, spiny
    vs aspiny, so the type grouping reads without a key.

    `mode` :
        'fraction'   — raw share of the member neurons. With `reference_cell_types`
                       the reference's own composition is drawn behind as an outlined
                       bar, which is what makes the share fair to read: 23P and 4P
                       dominate the members because they dominate the pool, and the
                       outline says so without the reader decoding a z
        'z'          — hypergeometric z of the member counts against
                       `reference_cell_types`; 0 is "as often as chance would pick it"
        'enrichment' — obs / expected on the same reference; 1 is chance
    'z' and 'enrichment' require `reference_cell_types` (see `member_cell_type_stats`).

    The reference should be the **recorded** population, not the whole column: the
    detector can only pick neurons that were imaged, so normalising to all column E
    cells would charge the coregistration's cell-type bias to the ensembles.

    `spiny_types_are_prefix=True` colours by the first len(spiny_types) entries of
    `order` instead of by name, for orderings whose leading types differ.

    `drop_types` leaves types out of the panel entirely — the unclassifiable ones
    ('WM-P', 'Unsure E') for figure 6. They come off the members *and* the reference,
    so both fractions are shares of the same restricted population and the outline
    stays comparable with the bars.

    Returns the plotted Series (index = `order`).
    """
    from matplotlib.ticker import PercentFormatter

    drop = set(drop_types)
    if drop:
        member_cell_type_df = member_cell_type_df[
            ~member_cell_type_df['cell_type'].isin(drop)]
        if reference_cell_types is not None:
            _ref = pd.Series(reference_cell_types)
            reference_cell_types = _ref[~_ref.isin(drop)]

    order = order if order is not None else member_cell_type_order(member_cell_type_df)
    order = [ct for ct in order if ct not in drop]
    if mode == 'fraction':
        d = member_cell_type_df.drop_duplicates('root_id')
        counts = d['cell_type'].value_counts().reindex(order).fillna(0)
        vals = counts / counts.sum()
        ylabel = 'Fraction of neurons' if ylabel is None else ylabel
    elif mode in ('z', 'enrichment'):
        if reference_cell_types is None:
            raise ValueError(f"mode={mode!r} needs reference_cell_types "
                             "(the cell_type Series of the recorded population)")
        stats = member_cell_type_stats(member_cell_type_df, reference_cell_types, order)
        vals = stats['z'] if mode == 'z' else stats['enrichment']
        if ylabel is None:
            ylabel = ('Enrichment vs recorded\npopulation (z)' if mode == 'z'
                      else 'Enrichment vs recorded\npopulation (obs / exp)')
    else:
        raise ValueError(f"mode must be 'fraction', 'z' or 'enrichment', got {mode!r}")

    if spiny_types_are_prefix:
        colors = [spiny_color if i < len(spiny_types) else aspiny_color
                  for i in range(len(order))]
    else:
        colors = [spiny_color if ct in set(spiny_types) else aspiny_color
                  for ct in order]

    x = np.arange(len(order))
    # Reference first, wider and unfilled, so the member bars sit inside it and the
    # comparison is a containment rather than two bars to line up by eye.
    if mode == 'fraction' and reference_cell_types is not None:
        ref = pd.Series(reference_cell_types).dropna()
        ref_frac = ref.value_counts().reindex(order).fillna(0) / len(ref)
        ax.bar(x, ref_frac, width=reference_width, facecolor='none',
               edgecolor=_NULL_COLOR if reference_color is None else reference_color,
               lw=1.4, label=reference_label)
    ax.bar(x, np.nan_to_num(np.asarray(vals, dtype=float)), width=bar_width, color=colors)
    # The chance level is the whole point of the normalised modes, so it gets a line
    # rather than being left to the reader's eye on the tick labels.
    if mode == 'z':
        ax.axhline(0, color='k', lw=1)
        # |z| > 1.96 is the only thing that separates a real bias in what the detector
        # picked from the wobble of drawing a few hundred neurons — without the band
        # every bar looks like a finding.
        if z_sig_level:
            for _s in (-z_sig_level, z_sig_level):
                ax.axhline(_s, color='0.6', lw=0.8, ls=':')
    elif mode == 'enrichment':
        ax.axhline(1, color='k', lw=1, ls='--')
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=label_rotation, ha='right')
    for tick, color in zip(ax.get_xticklabels(), colors):
        tick.set_color(color)
    ax.set_ylabel(ylabel)
    if mode == 'fraction':
        ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    else:
        ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))
    # The outline names itself through `reference_label`. The filled bars carry two
    # colours (spiny / aspiny), so no real bar handle can key them — `filled_label`
    # adds a neutral grey swatch that names the population they stand for instead.
    # They are the data, so that swatch leads and the outline they sit inside follows.
    handles, labels = ax.get_legend_handles_labels()
    if filled_label:
        from matplotlib.patches import Patch
        handles.insert(0, Patch(facecolor=filled_color, edgecolor='none'))
        labels.insert(0, filled_label)
    if legend and handles:
        ax.legend(handles, labels, frameon=False, loc='upper right')
    ax.spines[['top', 'right']].set_visible(False)
    return vals


def plot_cell_type_table(ax, table_df, spiny_color, aspiny_color,
                         spiny_types=SPINY_CELL_TYPES, columns=COVERAGE_TABLE_COLUMNS,
                         row_header='Cell type', total_label='All E',
                         fontsize=11, header_color='0.25', rule_color='0.3',
                         name_col_width=1.15, na_text='-', type_colors=None,
                         number_color='k'):
    """Draw `recorded_coverage_table`'s frame as a figure-quality table on `ax`.

    Rules only under the header and above the total row — a full grid is louder than
    the numbers. Type names take the same spiny / aspiny colours as the bar panels'
    tick labels, so the two read as the same grouping.

    `type_colors` overrides that colour for named types — the types figure 6 keeps in
    the table but drops from the bar panels are named in plain black, since the spiny /
    aspiny split does not mean anything for them. Only the name takes the colour; the
    numbers stay `number_color` throughout.
    """
    from matplotlib.table import Table

    ax.set_axis_off()
    spiny = set(spiny_types)

    def _fmt(v, f):
        return na_text if v is None or (isinstance(v, float) and np.isnan(v)) else f.format(v)

    rows   = list(table_df.index)
    n_rows = len(rows) + 1                      # + the header
    height = 1.0 / n_rows
    # Built cell by cell rather than through `ax.table`, so the name column can be
    # left-aligned and coloured per row while the numbers stay centred. Widths and
    # heights are relative — the bbox rescales them to fill the axes.
    table = Table(ax, bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)

    def _cell(r, c, width, text, loc, color):
        cell = table.add_cell(r, c, width=width, height=height, text=text, loc=loc,
                              edgecolor=rule_color, facecolor='none', fill=False)
        cell.set_linewidth(0.8)
        cell.set_text_props(color=color, fontsize=fontsize)
        # rule under the header and above the total row; a full grid is louder than
        # the numbers it would frame
        cell.visible_edges = ('B' if r == 0 else
                              'T' if rows[r - 1] == total_label else 'open')
        return cell

    _cell(0, 0, name_col_width, row_header, 'left', header_color)
    for c, (_k, head, _f) in enumerate(columns, start=1):
        _cell(0, c, 1.0, head, 'center', header_color)
    overrides = dict(type_colors or {})
    for r, name in enumerate(rows, start=1):
        color = (header_color if name == total_label else
                 spiny_color if name in spiny else aspiny_color)
        _cell(r, 0, name_col_width, str(name), 'left', overrides.get(name, color))
        for c, (key, _head, fmt) in enumerate(columns, start=1):
            _cell(r, c, 1.0, _fmt(table_df.loc[name, key], fmt), 'center', number_color)

    ax.add_table(table)
    return table

In [4]:
# ── what this figure needs, checked up front ─────────────────────────────────
# Neither is in the Zenodo snapshot; both are produced once, locally, in this order:
#   1. scripts/extract_calcium_data_via_docker.ipynb  → the two activity H5 files
#   2. python scripts/ensemble_run.py                 → the two run pickles
# Panel A's recorded-neuron list reads the calcium H5's key list only (no traces).
require_activity_h5(needed_by="figure S15's recorded-neuron list")
lds_path, ecker_path = require_ensemble_results('lds', 'ecker')

with open(lds_path, 'rb') as f:
    results = pickle.load(f)
with open(ecker_path, 'rb') as f:
    results_sup = pickle.load(f)

_neurons_df = load_neurons_table(use_column_manual_ct=True)

connectome neurons table:  1351
valid neurons w position: 1351


In [5]:
metric_f     = results['shared_input_strength']
metric_f_sup = results_sup['shared_input_strength']

# Per-neuron shared input strength for the whole filtered E population. It is a
# property of the filtered matrix, not of a neuron, so it needs the full
# calc_spines_features -> filter -> rebuild pass — cached across re-runs of this cell
# (`del SHARED_INPUT_SERIES` to force a recompute).
try:
    SHARED_INPUT_SERIES
except NameError:
    _syn_df_f = load_synapses_position_transformed(
        base_syn_table_path=CONNECTOME_SYN_TABLE_PATH)
    _df_f, _ = calc_spines_features(_neurons_df, _syn_df_f)
    SHARED_INPUT_SERIES = build_shared_input_strength_series(_df_f)

# Shared x ordering so the two cell-type panels line up column for column.
CT_ORDER = member_cell_type_order(results['member_cell_type_df'],
                                  results_sup['member_cell_type_df'])

# Reference for the cell-type panels: the *recorded* E neurons (calcium in >=1 scan),
# not the whole column — a type the detector never saw cannot be under-represented.
# Read off the H5 key list, so no traces are loaded.
RECORDED_EX_ROOT_IDS = recorded_ex_root_ids(_neurons_df)
RECORDED_CELL_TYPES  = _neurons_df.set_index('root_id').loc[RECORDED_EX_ROOT_IDS,
                                                            'cell_type']
print(f'Recorded E neurons: {len(RECORDED_EX_ROOT_IDS)} '
      f'(pickle says {results["n_recorded"]})')

# Recorded neurons against the whole column: the left panel of Fig 6 Sup 2, and the
# single-panel figure at the end of the notebook. Same two-column shape every cell-type
# panel takes (root_id + cell_type, one row per neuron); here the "members" are the
# recorded neurons and the reference is the column.
_col_ex_df = _neurons_df.loc[_neurons_df.clf_type == 'E', ['root_id', 'cell_type']]
_rec_ct_df = _col_ex_df[_col_ex_df.root_id.isin(RECORDED_EX_ROOT_IDS)]
CT_ORDER_COLUMN = member_cell_type_order(_col_ex_df)   # every E type, not just detected ones
REC_COVERAGE_TABLE = recorded_coverage_table(_rec_ct_df['cell_type'],
                                             _col_ex_df['cell_type'], CT_ORDER_COLUMN)
print('\nRecorded coverage per cell type:')
print(REC_COVERAGE_TABLE.round(3))

# Same reference population for the histogram baseline as for the cell-type outline:
# the recorded neurons. 11 of the 476 do not survive the spine filter and drop out.
_recorded_shared_input = SHARED_INPUT_SERIES.reindex(RECORDED_EX_ROOT_IDS).dropna()
RECORDED_SHARED_INPUT  = float(_recorded_shared_input.mean())
print(f'Shared input strength — recorded E: {RECORDED_SHARED_INPUT:.3f} '
      f'(n={len(_recorded_shared_input)})  |  all filtered E: '
      f'{SHARED_INPUT_SERIES.mean():.3f} (n={len(SHARED_INPUT_SERIES)})')
for _name, _m in (('LDS', metric_f), ('Ecker', metric_f_sup)):
    print(f'F  {_name:<6} obs={_m["obs"]:.3f}  null={_m["null_mean"]:.3f}  '
          f'{_m["fold"]:.3f}x  p={_m["p_emp"]:.4f} {_m["star"]}  '
          f'({_m["n_members"]} member slots)')
print(f'Cell-type order: {CT_ORDER}')
for _name, _ct_df in (('LDS', results['member_cell_type_df']),
                      ('Ecker', results_sup['member_cell_type_df'])):
    print(f'\n{_name} members vs recorded population '
          f'({_ct_df.root_id.nunique()} distinct neurons):')
    print(member_cell_type_stats(_ct_df, RECORDED_CELL_TYPES, CT_ORDER).round(3))


spine table incoming size (4567647, 4)
spine table outgoing size (819832, 4)
neurons: 1351
synapses with tags: 145356


100%|██████████| 1351/1351 [00:16<00:00, 82.65it/s]


Filtering neurons with valid spine data...
Remaining neurons after filtering: 1298
fixing networks
Filtering: Reducing matrix from 1351 to 1298 neurons.
Shared input strength: 1139 E neurons in the filtered population (median=174.9)
Recorded E neurons: 476 (pickle says 476)

Recorded coverage per cell type:
           n_reference  n_subset  coverage  ref_frac  subset_frac
cell_type                                                        
23P              349.0     191.0     0.547     0.294        0.401
4P               266.0     149.0     0.560     0.224        0.313
5P-IT            137.0      59.0     0.431     0.115        0.124
5P-NP             10.0       1.0     0.100     0.008        0.002
5P-PT             38.0      31.0     0.816     0.032        0.065
6P-CT            143.0      16.0     0.112     0.120        0.034
6P-IT            192.0      25.0     0.130     0.162        0.053
6P-U              28.0       3.0     0.107     0.024        0.006
WM-P              20.0       0.

In [6]:
# control font size
plt.rcParams['font.size'] = 19
plt.rcParams['legend.fontsize'] = 15
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.labelsize'] = 15
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.labelsize'] = 15
plt.rcParams['font.family'] = 'Arial'

LABEL_FONTSIZE   = 14   # schematic captions ('Ensemble' / 'Control' / 'Shared input' / …)
LEG_FONTSIZE     = 14   # inner legends in B / C / D
letter_font_size = 24   # 'A' 'B' 'C' 'D' 'E' panel letters
INSET_FONTSIZE   = 13   # Panel D — shared-E-input inset labels/ticks/star
FOLD_FONTSIZE    = 11.5   # Panel D — 'N.NNx' fold annotations above by-size bars
RASTER_FONTSIZE  = 14   # Panel A — every raster text (row labels, clip nums, scale bar, …)


In [7]:
# =============================================================================
# LAYOUT CONFIG - Fig 6 Sup 2.
#   left  (full height) : who got recorded - the 476 recorded E neurons against all
#                         column E, per cell type. Neither detector: the pool both of
#                         them had to pick from
#   right (two rows)    : one row per detector (LDS on top, Ecker below),
#                         histogram | cell types
# =============================================================================
SUP2_FIG_SIZE    = (18.5, 8.5)
SUP2_ROW_HEIGHTS = [1.0, 1.0]
SUP2_ROW_GAP     = 0.02   # the rows already clear each other: the top margin holds
                          # the legend and the bottom one the two-line x label

# Outer split: table | the two detector rows.
SUP2_MAIN_SPLIT  = [0.62, 2.0]
SUP2_MAIN_GAP    = 0.04

# Histogram column gets the extra width: its x label is two lines and its legend sits
# above the axes, so it needs the room the bar panel does not.
SUP2_COL_SPLIT   = [1.15, 1.0]
SUP2_COL_GAP     = 0.06

SUP2_TABLE_MARG  = dict(left=0.03, right=0.99, top=0.93, bottom=0.03)  # top: panel letter
SUP2_HIST_MARG   = dict(left=0.22, right=0.97, top=0.84, bottom=0.24)
SUP2_BAR_MARG    = dict(left=0.18, right=0.97, top=0.84, bottom=0.24)
SUP2_ROW_LABEL_X = 0.005          # row name ('LDS' / 'Ecker') in row-subfigure coords
SUP2_ROW_LABEL_FS = 20

SHARED_INPUT_LABEL = ('Shared input strength/neuron\n'
                      '(mean out-degree of all presynaptic neurons)')
SUP2_TABLE_FS      = 13

# The two unclassifiable E types: kept in the table (they are part of the column and
# of what was recorded) but named in black there, since the spiny / aspiny split does
# not mean anything for them, and left out of the member bar panels, where a type
# nobody can name says nothing about what the detector picked.
UNCLASSIFIED_CT       = ('WM-P', 'Unsure E')
SUP2_TABLE_CT_COLORS  = {ct: 'k' for ct in UNCLASSIFIED_CT}

SUP2_LETTER_XY  = (0.015, 0.99)   # panel letter inside each subfigure
SUP2_LETTER_FS  = 20              # matches Fig 6 Sup 1

# Cell-type panel: 'fraction' (share of the member neurons, with the recorded
# population outlined behind), or 'z' / 'enrichment' against that same reference.
# The numbers the other two modes would plot are printed by the cell above regardless.
CT_PANEL_MODE = 'fraction'
# The filled bars are two-coloured (spiny / aspiny), so they get a neutral grey
# swatch in the legend rather than a bar handle that could only stand for one colour.
MEMBER_FILL_LABEL = 'Ensemble population'
# =============================================================================

fig_sup2 = plt.figure(figsize=SUP2_FIG_SIZE, dpi=600)
sf_table, sf_rows = fig_sup2.subfigures(1, 2, width_ratios=SUP2_MAIN_SPLIT,
                                        wspace=SUP2_MAIN_GAP)

# ---- left, full height: recorded vs whole column, as numbers ----
ax_table = sf_table.subplots(1, 1)
sf_table.subplots_adjust(**SUP2_TABLE_MARG)
plot_cell_type_table(ax_table, REC_COVERAGE_TABLE,
                     spiny_color=SPINY_COLOR, aspiny_color=SHAFT_COLOR,
                     fontsize=SUP2_TABLE_FS, type_colors=SUP2_TABLE_CT_COLORS)
sf_table.text(*SUP2_LETTER_XY, 'A', fontsize=SUP2_LETTER_FS, fontweight='bold', va='top')

# ---- right: one row per detector ----
sfs_sup2 = sf_rows.subfigures(2, 1, height_ratios=SUP2_ROW_HEIGHTS, hspace=SUP2_ROW_GAP)

for _row, (_name, _mf, _ct_df) in enumerate((
        ('LDS',   metric_f,     results['member_cell_type_df']),
        ('Ecker', metric_f_sup, results_sup['member_cell_type_df']))):

    sf_hist, sf_ct = sfs_sup2[_row].subfigures(1, 2, width_ratios=SUP2_COL_SPLIT,
                                               wspace=SUP2_COL_GAP)
    if _row == 0:   # the letters name the columns, so only the top row carries them
        for _sf, _letter in ((sf_hist, 'B'), (sf_ct, 'C')):
            _sf.text(*SUP2_LETTER_XY, _letter, fontsize=SUP2_LETTER_FS,
                     fontweight='bold', va='top')

    # ---- Metric F null vs observed ----
    ax_hist = sf_hist.subplots(1, 1)
    sf_hist.subplots_adjust(**SUP2_HIST_MARG)
    plot_null_hist_panel(
        ax_hist, _mf['valid_null'],
        obs_val=_mf['obs'], global_val=RECORDED_SHARED_INPUT,
        ctrl_color=CONTROL_COLOR, obs_color=ex_color,
        obs_label=f'Ensembles ({_mf["obs"]:.1f})',
        global_label=f'Recorded population ({RECORDED_SHARED_INPUT:.0f})',
        ctrl_mean_fmt='.1f', star=_mf['star'], headroom=1.9,   # star clears the legend
        legend_bbox=(0, 1.125), bins=30, xlabel=SHARED_INPUT_LABEL)
    ax_hist.xaxis.set_major_formatter(FormatStrFormatter('%g'))

    # ---- cell types of the member neurons ----
    ax_ct = sf_ct.subplots(1, 1)
    sf_ct.subplots_adjust(**SUP2_BAR_MARG)
    plot_member_cell_type_panel(
        ax_ct, _ct_df, spiny_color=SPINY_COLOR, aspiny_color=SHAFT_COLOR,
        order=CT_ORDER, mode=CT_PANEL_MODE,
        reference_cell_types=RECORDED_CELL_TYPES,
        filled_label=MEMBER_FILL_LABEL, drop_types=UNCLASSIFIED_CT,
        legend=(_row == 0))   # one row names the bars; both rows use the same pair

    sfs_sup2[_row].text(SUP2_ROW_LABEL_X, 0.5, _name, fontsize=SUP2_ROW_LABEL_FS,
                        va='center', ha='left')

plt.savefig('fig_s15.pdf', format='pdf', bbox_inches='tight')
